In [3]:
import os
import shutil
from typing import List, Tuple, Optional
from pathlib import Path
from typing import List
import pandas as pd
import lightgbm

def imputer(df, target_columns=None, lgbm_params=None):
    """
    Impute missing values in a time-series Series or DataFrame.
    Here we apply a linearly interpolation on 2 consecutive NaNs or fewer.
    If there is a larger chunk of NaNs then we use a LightGBM regressor to fill in longer missing intervals.

    Parameters
    ----------
    df : pd.Series or pd.DataFrame
        A pandas Series or DataFrame with a DatetimeIndex.
    target_columns : list or None
        Which columns to impute. If None, all columns with NaNs will be imputed.
        (Ignored if df is a Series, because there's only one column.)
    lgbm_params : dict or None
        Parameters for the LGBMRegressor. If None, defaults are used.

    Returns
    -------
    pd.Series or pd.DataFrame
        The same type as `df`, with missing values imputed.
    """
    # Check if input is a Series
    was_series = False
    original_name = None
    if isinstance(df, pd.Series):
        was_series = True
        original_name = df.name if df.name is not None else "value"
        df = df.to_frame(name=original_name)
    
    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("DataFrame/Series must have a DatetimeIndex.")
    
    # Identify columns needing imputation
    if target_columns is None:
        target_columns = [col for col in df.columns if df[col].isna().any()]
    
    # First pass: linearly interpolate small gaps (<= 2 consecutive NaNs)
    for col in target_columns:
        small_gaps = df[col].isna().astype(int).groupby(df[col].notna().cumsum()).sum()
        if any((small_gaps <= 2) & (small_gaps > 0)):
            print(f"Column '{col}' has small gaps (<= 2 consecutive NaNs).")
    df[target_columns] = df[target_columns].interpolate(method='linear', limit=2)
    
    # Default LightGBM parameters if none provided
    if lgbm_params is None:
        lgbm_params = {
            'n_estimators': 500,
            'learning_rate': 0.1,
            'random_state': 42
        }

    # Create time-based features
    global_time_features = pd.DataFrame(index=df.index)
    global_time_features['minute'] = df.index.minute
    global_time_features['hour'] = df.index.hour
    global_time_features['day'] = df.index.day
    global_time_features['dayofweek'] = df.index.dayofweek
    global_time_features['month'] = df.index.month

    #LightGBM for larger gaps
    for target in target_columns:
        # Check if there are still missing values after small-gap interpolation
        if not df[target].isna().any():
            continue 
        print(f"Column '{target}' has large gaps (> 2 consecutive NaNs).")

        # we only train the model on rows for which the target column has non‐missing values
        not_null_mask = df[target].notna()
        y_train = df.loc[not_null_mask, target]

        # Build training features
        other_cols = df.columns.difference([target])
        X_train = pd.concat([df.loc[not_null_mask, other_cols],
                             global_time_features.loc[not_null_mask]], axis=1)

        # Train LightGBM
        model = lightgbm.LGBMRegressor(**lgbm_params,verbose=-1)
        model.fit(X_train, y_train)

        # Predict for the missing rows
        null_mask = df[target].isna()
        if null_mask.any():
            X_missing = pd.concat([df.loc[null_mask, other_cols],
                                   global_time_features.loc[null_mask]], axis=1)
            df.loc[null_mask, target] = model.predict(X_missing)

    if was_series:
        col_name = df.columns[0]
        df_series = df[col_name]
        df_series.name = original_name
        return df_series

    return df

def feature_correlation_sorter_and_remover(df: pd.DataFrame, target: str, threshold: float = 0.1) -> pd.Series:
    """
    This function sorts the features based on their correlation with the target 
    and removes the ones that are below the threshold.
    
    Parameters:
        df (pd.DataFrame): The DataFrame with the features and the target column.
        target (str): The name of the target column.
        threshold (float): The threshold for the correlation (default is 0.1).
    
    Returns:
        pd.Series: The features (rows) that are above the threshold and their correlation with the target.
    """
    # Compute correlation with the target column
    correlation_with_target = df.corr(method="pearson")[target].abs()
    
    # Sort features by correlation values in descending order
    sorted_features = correlation_with_target.sort_values(ascending=False)
    
    # Filter features based on the threshold
    filtered_features = sorted_features[sorted_features >= threshold]

    # Return the filtered and sorted features with their correlations
    return filtered_features.drop(target, errors='ignore')


def list_creator_flagger(
    df: pd.DataFrame, 
    substrings: Optional[List[str]] = None
) -> Tuple[List[str], List[str]]:
    """
    Separates DataFrame columns into two lists: those containing specified substrings 
    and those that do not.

    Parameters:
        df (pd.DataFrame): The DataFrame with the features.
        substrings (List[str], optional): The list of substrings to search for in column names.
            Default is ['flag', 'cos', 'sin', 'day_of_week', 'day_of_month', 'weekend', 
                       'days_in_month', 'hour', 'minute'].

    Returns:
        - non_flag_columns (List[str]): Columns that do not contain the specified substrings.
        - flag_columns (List[str]): Columns that contain the specified substrings.
    """
    if substrings is None:
        substrings = [
            'flag', 'cos', 'sin', 'day_of_week', 'day_of_month', 
            'weekend', 'days_in_month', 'hour', 'minute'
        ]

    flag_columns = [col for col in df.columns if any(substring in col for substring in substrings)]

    if not flag_columns:
        print("No columns with the specified substrings found.")
        return [], []

    non_flag_columns = [col for col in df.columns if col not in flag_columns]

    return non_flag_columns, flag_columns

def empty_folder(folder: str) -> None:
    """
    Empties a folder by deleting all its contents. If the folder does not exist, it creates it.

    Args:
        folder (str): The path to the folder to be emptied.
    """
    # Get the absolute path of the folder
    absolute_folder_path = os.path.abspath(folder)
    print(f"Attempting to empty folder: {absolute_folder_path}")
    
    # Check if the directory exists; create it if not
    if not os.path.exists(absolute_folder_path):
        os.makedirs(absolute_folder_path)  # Creates the folder if it doesn't exist
        print(f"Folder did not exist and was created: {absolute_folder_path}")
        return  # Exit after creating, no need to empty

    print(f"Folder exists and will be emptied: {absolute_folder_path}")
        
    # Iterate over the items in the directory
    for item in os.listdir(absolute_folder_path):
        item_path = os.path.join(absolute_folder_path, item)
        print(f"Processing item: {item_path}")
        
        # Delete file or directory
        try:
            if os.path.isfile(item_path):
                os.remove(item_path)
                print(f"Deleted file: {item_path}")
            elif os.path.isdir(item_path):
                shutil.rmtree(item_path)
                print(f"Deleted directory: {item_path}")
        except Exception as e:
            print(f"Error deleting {item_path}: {e}")
    
    print(f"The folder '{folder}' has been emptied.")

def save_best_params_to_csv(study, path: str, hours: int) -> None:
    """
    Save the best parameters from an Optuna study to a CSV file in the 'models' directory.
    Every time the function is called, it appends the new data to the existing CSV file.

    Args:
        study: Optuna study object containing the best parameters and trial results.
        path (str): Base path where the 'models' folder is located.
        hours (int): Multiplication factor for 'hours_in'.
    """
    # Extract best parameters and additional information
    best_params = study.best_params.copy()
    best_params["mse"] = study.best_trial.value
    best_params["trials"] = len(study.trials)
    best_params['hours_in'] *= hours  # Convert back to points

    # Handle `fc_layers` (convert to string for saving)
    fc_layers = [
        best_params.pop(f"fc_layer_{i+1}", None) for i in range(best_params.get("n_layers", 0))
    ]
    best_params["fc_layers"] = ",".join(map(str, fc_layers))  # Save as comma-separated string

    # Define the column names
    columns = [
        'hours_in', 'hours_out', 'suggested_hidden_dim', 'n_rnn_layers',
        'dropout', 'lr','fc_layers', 'mse', 'trials'
    ]

    # Create a DataFrame for the best parameters
    best_params_df = pd.DataFrame([best_params])

    # Define the file path
    target_dir = Path(path) / 'models'
    hyper_params_filepath = target_dir / "hyper_params_df.csv"
    print(f"Saving to file: {hyper_params_filepath}")

    # Check if the file exists
    if hyper_params_filepath.is_file():
        # Read the existing file
        existing_df = pd.read_csv(hyper_params_filepath)
        print("File exists. Appending data.")
        
        if set(columns).issubset(existing_df.columns):
            # Append new data to existing DataFrame
            updated_df = pd.concat([existing_df, best_params_df], ignore_index=True)
        else:
            print("Warning: CSV columns do not match expected columns. Columns will be reset.")
            # Create a new DataFrame with the correct columns
            updated_df = pd.DataFrame(columns=columns)
            updated_df = pd.concat([updated_df, best_params_df], ignore_index=True)
    else:
        print("File does not exist. Creating new file.")
        # Create a new DataFrame with the correct columns
        updated_df = pd.DataFrame(columns=columns)
        updated_df = pd.concat([updated_df, best_params_df], ignore_index=True)

    # Save the DataFrame to CSV
    updated_df.to_csv(hyper_params_filepath, index=False)
    print("Data saved successfully.")
    
def read_hyper_params(path: str) -> dict:
    """
    Reads the hyperparameters from the CSV file and returns the last line as a dictionary.

    Args:
        path (str): The path to the folder where the models are stored.

    Returns:
        dict: The last line of the CSV file as a dictionary.
    
    Raises:
        ValueError: If the CSV file is empty.
    """
    # Define the path to the CSV file
    path_csv = Path(path) / "models" / "hyper_params_df.csv"
    
    # Read the CSV file into a DataFrame
    df = pd.read_csv(path_csv)
    
    if df.empty:
        raise ValueError("The CSV file is empty.")
    
    # Get the last row as a Series
    last_line = df.iloc[-1]
    
    # Convert the Series to a dictionary
    last_line_dict = last_line.to_dict()
    
    # Process `fc_layers` (convert back to a list of integers)
    if "fc_layers" in last_line_dict:
        last_line_dict["fc_layers"] = list(map(int, last_line_dict["fc_layers"].split(",")))

    # Define the keys that should not be converted to int
    exclude_from_int_conversion = {"dropout", "lr", "fc_layers"}
    
    # Process the dictionary
    processed_dict = {}
    for key, value in last_line_dict.items():
        if key in exclude_from_int_conversion:
            processed_dict[key] = value
        else:
            try:
                processed_dict[key] = int(value)
            except (ValueError, TypeError):
                processed_dict[key] = value
    
    return processed_dict

def separate_past_and_future_features(ordered_features_list):
    """
    Separates features into past and future covariates based on specific conditions.

    Args:
        ordered_features_list (list of str): List of feature names in the desired order.

    Returns:
        tuple: A tuple containing two lists:
            - past_covariates (list of str): Features considered as past covariates.
            - future_covariates (list of str): Features considered as future covariates.
    """
    past_covariates = []
    future_covariates = []
    
    future_condition_strings = [
        "sin", "cos", "hour", "flag", "day", "month", "week",
        "weekend", "day_of_week", "day_of_month", "days_in_month",
        "minute"
    ]
    
    for feature in ordered_features_list:
        if any(condition_string in feature for condition_string in future_condition_strings):
            future_covariates.append(feature)
        else:
            past_covariates.append(feature)
    
    return past_covariates, future_covariates



c:\Users\Dimitrios\environments\test_grecco\Lib\site-packages\dask\dataframe\__init__.py:42: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


# import files 

In [5]:
from pathlib import Path
import sys
import pandas as pd

# Paths
project_path = Path(r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-")

meteo_path = project_path / "inputs" / "raw" / "meteo.csv"
functions_path = project_path / "scripts" / "data handling"

# Import your imputer function
sys.path.append(str(functions_path))
from data_handling_functions import imputer

# Read meteo.csv
meteo_df = pd.read_csv(meteo_path)

print(meteo_df.head())
print(meteo_df.columns.tolist())


          Date_Time  Temp__C  HighTemp__C  LowTemp__C  Hum__  HighHum__  \
0  01.01.2025 00:00     -3.4         -3.4        -3.4   94.6       94.6   
1  01.01.2025 00:05     -3.4         -3.4        -3.4   94.6       94.6   
2  01.01.2025 00:10     -3.4         -3.4        -3.4   94.6       94.6   
3  01.01.2025 00:15     -3.4         -3.3        -3.4   94.6       94.6   
4  01.01.2025 00:20     -3.4         -3.3        -3.4   94.6       94.6   

   LowHum__  DewPoint__C  HighDewPoint__C  LowDewPoint__C  ...  Rain_Mm  \
0      94.6         -4.2             -4.2            -4.2  ...      0.0   
1      94.6         -4.1             -4.1            -4.2  ...      0.0   
2      94.6         -4.1             -4.1            -4.2  ...      0.0   
3      94.6         -4.1             -4.1            -4.2  ...      0.0   
4      94.6         -4.1             -4.1            -4.1  ...      0.0   

   HighRainRate_Mm_h  SolarRad_W_m_2  HighSolarRad_W_m_2  UVIndex  \
0                0.0         

In [6]:
meteo_df = pd.read_csv(meteo_path)

meteo_df["Date_Time"] = pd.to_datetime(
    meteo_df["Date_Time"],
    format="%d.%m.%Y %H:%M"
)

meteo_df = meteo_df.set_index("Date_Time").sort_index()

print(meteo_df.isna().sum())

Temp__C               166
HighTemp__C           166
LowTemp__C            166
Hum__                 166
HighHum__             166
LowHum__              166
DewPoint__C           167
HighDewPoint__C       167
LowDewPoint__C        167
WetBulb__C            166
HighWetBulb__C        169
LowWetBulb__C         166
AvgWindSpeed_M_s      169
HighWindSpeed_M_s     170
WindChill__C          178
LowWindChill__C       178
HeatIndex__C          174
HighHeatIndex__C      174
THWIndex__C           179
HighTHWIndex__C       179
LowTHWIndex__C        179
WindRun_M             169
THSWIndex__C          184
HighTHSWIndex__C      184
LowTHSWIndex__C       184
Rain_Mm               166
HighRainRate_Mm_h     169
SolarRad_W_m_2        175
HighSolarRad_W_m_2    175
UVIndex               167
HighUVIndex           167
UVDose_MEDs           167
HeatingDegreeDays     166
CoolingDegreeDays     166
Quality                 0
dtype: int64


In [7]:
meteo_imputed = imputer(meteo_df)

print(meteo_imputed.isna().sum())

Column 'Temp__C' has small gaps (<= 2 consecutive NaNs).
Column 'HighTemp__C' has small gaps (<= 2 consecutive NaNs).
Column 'LowTemp__C' has small gaps (<= 2 consecutive NaNs).
Column 'Hum__' has small gaps (<= 2 consecutive NaNs).
Column 'HighHum__' has small gaps (<= 2 consecutive NaNs).
Column 'LowHum__' has small gaps (<= 2 consecutive NaNs).
Column 'DewPoint__C' has small gaps (<= 2 consecutive NaNs).
Column 'HighDewPoint__C' has small gaps (<= 2 consecutive NaNs).
Column 'LowDewPoint__C' has small gaps (<= 2 consecutive NaNs).
Column 'WetBulb__C' has small gaps (<= 2 consecutive NaNs).
Column 'HighWetBulb__C' has small gaps (<= 2 consecutive NaNs).
Column 'LowWetBulb__C' has small gaps (<= 2 consecutive NaNs).
Column 'AvgWindSpeed_M_s' has small gaps (<= 2 consecutive NaNs).
Column 'HighWindSpeed_M_s' has small gaps (<= 2 consecutive NaNs).
Column 'WindChill__C' has small gaps (<= 2 consecutive NaNs).
Column 'LowWindChill__C' has small gaps (<= 2 consecutive NaNs).
Column 'HeatI

In [8]:
print(meteo_imputed.isna().sum())

Temp__C               0
HighTemp__C           0
LowTemp__C            0
Hum__                 0
HighHum__             0
LowHum__              0
DewPoint__C           0
HighDewPoint__C       0
LowDewPoint__C        0
WetBulb__C            0
HighWetBulb__C        0
LowWetBulb__C         0
AvgWindSpeed_M_s      0
HighWindSpeed_M_s     0
WindChill__C          0
LowWindChill__C       0
HeatIndex__C          0
HighHeatIndex__C      0
THWIndex__C           0
HighTHWIndex__C       0
LowTHWIndex__C        0
WindRun_M             0
THSWIndex__C          0
HighTHSWIndex__C      0
LowTHSWIndex__C       0
Rain_Mm               0
HighRainRate_Mm_h     0
SolarRad_W_m_2        0
HighSolarRad_W_m_2    0
UVIndex               0
HighUVIndex           0
UVDose_MEDs           0
HeatingDegreeDays     0
CoolingDegreeDays     0
Quality               0
dtype: int64


In [9]:
from pathlib import Path

clean_folder = project_path / "inputs" / "clean"
clean_folder.mkdir(parents=True, exist_ok=True)

output_path = clean_folder / "meteo.csv"

meteo_imputed.to_csv(output_path)

print(f"Saved to: {output_path}")

Saved to: C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\inputs\clean\meteo.csv


# for the load

In [ ]:
import pandas as pd

path = r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\inputs\raw\Consumption_dataset_1min.csv"

df = pd.read_csv(path)

print(df.head())
print(df.columns.tolist())
print(df.shape)

          timestamp  avg_power  Quality
0  01.01.2025 00:00   4098.504        0
1  01.01.2025 00:01   2173.104        0
2  01.01.2025 00:02   2170.962        0
3  01.01.2025 00:03   2174.946        0
4  01.01.2025 00:04   4153.698        0
['timestamp', 'avg_power', 'Quality']
(525600, 3)


In [11]:
df = pd.read_csv(path)

# Parse timestamp
df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    format="%d.%m.%Y %H:%M"
)

# Set datetime index
df = df.set_index("timestamp").sort_index()

# Remove Quality column
df = df.drop(columns=["Quality"])

# Check missing values
print(df.isna().sum())

avg_power    0
dtype: int64


In [12]:
print(df.isna().sum())

avg_power    0
dtype: int64


In [14]:
print(df.columns)
print(df.index)

Index(['avg_power'], dtype='str')
DatetimeIndex(['2025-01-01 00:00:00', '2025-01-01 00:01:00',
               '2025-01-01 00:02:00', '2025-01-01 00:03:00',
               '2025-01-01 00:04:00', '2025-01-01 00:05:00',
               '2025-01-01 00:06:00', '2025-01-01 00:07:00',
               '2025-01-01 00:08:00', '2025-01-01 00:09:00',
               ...
               '2025-12-31 23:50:00', '2025-12-31 23:51:00',
               '2025-12-31 23:52:00', '2025-12-31 23:53:00',
               '2025-12-31 23:54:00', '2025-12-31 23:55:00',
               '2025-12-31 23:56:00', '2025-12-31 23:57:00',
               '2025-12-31 23:58:00', '2025-12-31 23:59:00'],
              dtype='datetime64[us]', name='timestamp', length=525600, freq=None)


In [15]:
df = df.sort_index()

expected = pd.date_range(
    start=df.index.min(),
    end=df.index.max(),
    freq="1min"
)

print("Sorted:", df.index.is_monotonic_increasing)
print("Duplicate timestamps:", df.index.duplicated().sum())
print("Missing timestamps:", len(expected.difference(df.index)))
print("Extra timestamps:", len(df.index.difference(expected)))
print("First:", df.index.min())
print("Last :", df.index.max())

Sorted: True
Duplicate timestamps: 0
Missing timestamps: 0
Extra timestamps: 0
First: 2025-01-01 00:00:00
Last : 2025-12-31 23:59:00


In [16]:
df

,avg_power
timestamp,
2025-01-01 00:00:00,4098.504
2025-01-01 00:01:00,2173.104
2025-01-01 00:02:00,2170.962
2025-01-01 00:03:00,2174.946
2025-01-01 00:04:00,4153.698
...,...
2025-12-31 23:55:00,4074.630
2025-12-31 23:56:00,2174.628
2025-12-31 23:57:00,2151.480


In [ ]:
df.to_csv(
    r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\inputs\clean\Consumption_dataset_1min.csv"
)

# final check

In [19]:
import pandas as pd

files = {
    "consumption": r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\inputs\clean\Consumption_dataset_1min_imputed.csv",
    "meteo": r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\inputs\clean\meteo_imputed.csv",
}

for name, path in files.items():

    print("\n" + "="*80)
    print(name.upper())
    print("="*80)

    df = pd.read_csv(path)

    # Find timestamp column
    time_col = None
    for c in df.columns:
        if c.lower() in ["timestamp", "date_time", "datetime", "date"]:
            time_col = c
            break

    print("Timestamp column:", time_col)

    df[time_col] = pd.to_datetime(df[time_col])

    # Sort checks
    print("Sorted:", df[time_col].is_monotonic_increasing)

    # Duplicate timestamps
    print("Duplicate timestamps:", df[time_col].duplicated().sum())

    # Missing values
    print("\nNaNs per column:")
    print(df.isna().sum()[df.isna().sum() > 0])

    # Missing timestamps
    freq = pd.infer_freq(df[time_col])

    print("\nInferred frequency:", freq)

    if freq is not None:
        expected = pd.date_range(
            start=df[time_col].min(),
            end=df[time_col].max(),
            freq=freq
        )

        missing_ts = len(expected.difference(df[time_col]))

        print("Expected rows :", len(expected))
        print("Actual rows   :", len(df))
        print("Missing stamps:", missing_ts)

    print("\nFirst timestamp:", df[time_col].min())
    print("Last timestamp :", df[time_col].max())


CONSUMPTION
Timestamp column: timestamp
Sorted: True
Duplicate timestamps: 0

NaNs per column:
Series([], dtype: int64)

Inferred frequency: min
Expected rows : 525600
Actual rows   : 525600
Missing stamps: 0

First timestamp: 2025-01-01 00:00:00
Last timestamp : 2025-12-31 23:59:00

METEO
Timestamp column: Date_Time
Sorted: True
Duplicate timestamps: 0

NaNs per column:
Series([], dtype: int64)

Inferred frequency: 5min
Expected rows : 105120
Actual rows   : 105120
Missing stamps: 0

First timestamp: 2025-01-01 00:00:00
Last timestamp : 2025-12-31 23:55:00


# find 5 dates

In [21]:
import json
import pandas as pd
from pathlib import Path

# ============================================================
# PATHS
# ============================================================
project_path = Path(
    r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-"
)

input_path = project_path / "inputs" / "clean" / "Consumption_dataset_1min_imputed.csv"
output_path = project_path / "inputs" / "clean" / "forecast_dates.json"

# ============================================================
# READ DATA
# ============================================================
df = pd.read_csv(input_path)

df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

# ============================================================
# SETTINGS
# ============================================================
min_history_days = 30

winter_months = [1, 2, 12]
summer_months = [6, 7, 8]

first_allowed_date = df["timestamp"].min() + pd.Timedelta(days=min_history_days)
last_available_date = df["timestamp"].max().normalize()

# one forecast start per selected day at 00:00
available_days = pd.date_range(
    start=first_allowed_date.normalize(),
    end=last_available_date,
    freq="D"
)

# keep only days that actually exist in the dataset
existing_days = set(df["timestamp"].dt.normalize())

available_days = [
    day for day in available_days
    if day in existing_days
]

# ============================================================
# SELECT DATES
# ============================================================
winter_days = [
    day for day in available_days
    if day.month in winter_months
]

summer_days = [
    day for day in available_days
    if day.month in summer_months
]

# evenly select 5 dates from each season
def select_evenly(days, n=5):
    if len(days) < n:
        raise ValueError(f"Only {len(days)} available days, but {n} requested.")
    
    idx = pd.Series(range(len(days))).quantile(
        [i / (n - 1) for i in range(n)]
    ).round().astype(int).tolist()
    
    return [days[i] for i in idx]

selected_winter = select_evenly(winter_days, 5)
selected_summer = select_evenly(summer_days, 5)

# ============================================================
# EXPORT JSON
# ============================================================
forecast_dates = {
    "winter": {
        f"day{i+1}": selected_winter[i].strftime("%Y-%m-%d 00:00:00")
        for i in range(5)
    },
    "summer": {
        f"day{i+1}": selected_summer[i].strftime("%Y-%m-%d 00:00:00")
        for i in range(5)
    }
}

with open(output_path, "w") as f:
    json.dump(forecast_dates, f, indent=4)

print(json.dumps(forecast_dates, indent=4))
print(f"\nSaved to: {output_path}")

{
    "winter": {
        "day1": "2025-01-31 00:00:00",
        "day2": "2025-02-15 00:00:00",
        "day3": "2025-12-02 00:00:00",
        "day4": "2025-12-16 00:00:00",
        "day5": "2025-12-31 00:00:00"
    },
    "summer": {
        "day1": "2025-06-01 00:00:00",
        "day2": "2025-06-24 00:00:00",
        "day3": "2025-07-17 00:00:00",
        "day4": "2025-08-08 00:00:00",
        "day5": "2025-08-31 00:00:00"
    }
}

Saved to: C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\inputs\clean\forecast_dates.json
